# Prospective Metric Pairs Analysis: hdEEG

## Objective

Test all possible combinations of pre-stimulus and post-stimulus metrics to identify which metric pairs provide the strongest prospective closed-loop effects.

## Method

For each metric pair (pre, post):

1. **Training Phase:** Learn threshold on first 50% of trials (median pre-stimulus metric)
2. **Test Phase:** Apply threshold prospectively to second 50% of trials
3. **Evaluation:** Measure post-stimulus metric variability reduction

## Metrics

Report for each pair:
- % sessions with positive effect size
- % sessions with improvement > baseline
- % sessions with statistical significance (p < 0.05)
- Mean effect size

In [9]:
from pathlib import Path
import pandas as pd
import numpy as np
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

print('Libraries loaded.')

Libraries loaded.


In [10]:
# Locate project root
cwd = Path.cwd().resolve()
if (cwd / 'data').exists():
    project_root = cwd
elif (cwd.parent / 'data').exists():
    project_root = cwd.parent
else:
    raise FileNotFoundError('Could not find project root containing data/.')

metrics_root = project_root / 'data' / 'df_results' / 'hd-eeg_metrics'

# Load data
csv_path = metrics_root / 'df_9MOIs_hd-eeg.csv'
df = pd.read_csv(csv_path)

print(f'Data loaded from: {csv_path}')
print(f'Total rows: {df.shape[0]}')

# Get unique metrics
unique_metrics = sorted(df['metric_name'].unique())
print(f'\nAvailable metrics ({len(unique_metrics)}): {unique_metrics}')

# Use radius 100 only
RADIUS_PRE = 100
RADIUS_POST = 100

Data loaded from: /Users/cbc/Documents/GitHub/fufo/notebook/DavideMomi/Revision/State_Dependent_Brain_Stimulation-main/data/df_results/hd-eeg_metrics/df_9MOIs_hd-eeg.csv
Total rows: 2962017

Available metrics (9): ['Entropy_FC', 'N1', 'N1_t', 'N2', 'N2_t', 'dynamic_functional_connectivity_matrix_var_mat', 'phase_coherence_matrix_mean_mat', 'salience', 'zero_crossing_rate']


In [11]:
def prospective_closed_loop_test(pre_vals, post_vals):
    """
    Within-session prospective closed-loop test:
    - Learn threshold on training trials (first 50%)
    - Apply to test trials (second 50%)
    - Measure if selection reduces post-stimulus variability
    
    Returns: dict with effect_size, p_value, spreads, and sample sizes
    """
    pre_vals = np.asarray(pre_vals).flatten()
    post_vals = np.asarray(post_vals).flatten()
    
    # Remove NaN pairs
    mask = ~(np.isnan(pre_vals) | np.isnan(post_vals))
    pre_vals = pre_vals[mask]
    post_vals = post_vals[mask]
    
    # Minimum trials requirement
    if len(pre_vals) < 8:
        return {k: np.nan for k in ['spread_favorable', 'spread_unfavorable', 'spread_baseline',
                                      'effect_size', 'p_value', 'n_favorable', 'n_unfavorable',
                                      'n_baseline', 'improvement_favorable']}
    
    # Split: training (first 50%) and test (second 50%)
    split_idx = len(pre_vals) // 2
    pre_train = pre_vals[:split_idx]
    pre_test = pre_vals[split_idx:]
    post_test = post_vals[split_idx:]
    
    # Learn threshold on training set (median pre-stimulus)
    threshold = np.median(pre_train)
    
    # Apply to test set
    favorable_mask = pre_test < threshold
    post_favorable = post_test[favorable_mask]
    post_unfavorable = post_test[~favorable_mask]
    
    # Measure spreads
    spread_favorable = np.std(post_favorable) if len(post_favorable) > 1 else np.nan
    spread_unfavorable = np.std(post_unfavorable) if len(post_unfavorable) > 1 else np.nan
    spread_baseline = np.std(post_test)
    
    # Effect size: normalized reduction
    if np.isnan(spread_favorable) or np.isnan(spread_unfavorable) or spread_baseline == 0:
        effect_size = np.nan
    else:
        effect_size = (spread_unfavorable - spread_favorable) / spread_baseline
    
    # Improvement over baseline
    if spread_baseline == 0:
        improvement_favorable = np.nan
    else:
        improvement_favorable = (spread_baseline - spread_favorable) / spread_baseline
    
    # T-test: absolute deviations from group means
    if len(post_favorable) > 1 and len(post_unfavorable) > 1:
        dev_favorable = np.abs(post_favorable - np.mean(post_favorable))
        dev_unfavorable = np.abs(post_unfavorable - np.mean(post_unfavorable))
        t_stat, p_value = stats.ttest_ind(dev_favorable, dev_unfavorable)
    else:
        p_value = np.nan
    
    return {
        'spread_favorable': spread_favorable,
        'spread_unfavorable': spread_unfavorable,
        'spread_baseline': spread_baseline,
        'effect_size': effect_size,
        'p_value': p_value,
        'n_favorable': np.sum(favorable_mask),
        'n_unfavorable': np.sum(~favorable_mask),
        'n_baseline': len(post_test),
        'improvement_favorable': improvement_favorable,
    }

print('Prospective closed-loop test function defined.')

Prospective closed-loop test function defined.


In [12]:
# Loop over all metric pairs
results_all_pairs = []

for metric_pre in unique_metrics:
    for metric_post in unique_metrics:
        # Filter for pre-stimulus metric
        df_pre = df[
            (df['radius_pre'] == RADIUS_PRE) & 
            (df['metric_name'] == metric_pre)
        ].copy()
        
        # Filter for post-stimulus metric
        df_post = df[
            (df['radius_post'] == RADIUS_POST) & 
            (df['metric_name'] == metric_post)
        ].copy()
        
        # Skip if either metric is missing
        if len(df_pre) == 0 or len(df_post) == 0:
            continue
        
        # Merge pre and post on (sub, run_is, trial_id) to align data
        df_filt = pd.merge(
            df_pre[['sub', 'run_is', 'trial_id', 'metric_value_pre', 'metric_value_post']].rename(
                columns={'metric_value_pre': 'pre_stim', 'metric_value_post': 'pre_stim_post'}),
            df_post[['sub', 'run_is', 'trial_id', 'metric_value_pre', 'metric_value_post']].rename(
                columns={'metric_value_pre': 'post_stim_pre', 'metric_value_post': 'post_stim'}),
            on=['sub', 'run_is', 'trial_id'],
            how='inner'
        )
        
        # Skip if merge resulted in no data
        if len(df_filt) == 0:
            continue
        
        # Run prospective test for all sessions
        results = []
        
        for (sub, run), session_df in df_filt.groupby(['sub', 'run_is']):
            session_df = session_df.sort_values('trial_id').reset_index(drop=True)
            
            # Extract pre-stimulus (used for threshold) and post-stimulus (used for outcome)
            pre_vals = session_df.groupby('trial_id')['pre_stim'].mean().values
            post_vals = session_df.groupby('trial_id')['post_stim'].mean().values
            
            test_result = prospective_closed_loop_test(pre_vals, post_vals)
            results.append(test_result)
        
        df_prospective = pd.DataFrame(results)
        df_prospective_valid = df_prospective.dropna(subset=['effect_size'])
        
        # Skip if no valid results
        if len(df_prospective_valid) == 0:
            continue
        
        # Summary statistics
        pct_positive = (df_prospective_valid['effect_size'] > 0).mean() * 100
        pct_improvement = (df_prospective_valid['improvement_favorable'] > 0).mean() * 100
        pct_significant = (df_prospective_valid['p_value'] < 0.05).sum() / len(df_prospective_valid) * 100
        
        mean_effect = df_prospective_valid['effect_size'].mean()
        mean_improvement = df_prospective_valid['improvement_favorable'].mean()
        
        n_sessions = len(df_prospective_valid)
        n_total = len(df_prospective)
        
        results_all_pairs.append({
            'metric_pre': metric_pre,
            'metric_post': metric_post,
            'n_sessions': n_sessions,
            'n_total': n_total,
            'pct_positive': pct_positive,
            'pct_improvement': pct_improvement,
            'pct_significant': pct_significant,
            'mean_effect': mean_effect,
            'mean_improvement': mean_improvement
        })

df_results = pd.DataFrame(results_all_pairs)

print(f'\nAnalysis complete!')
print(f'Total metric pairs tested: {len(df_results)}')


Analysis complete!
Total metric pairs tested: 81


In [13]:
# Display results sorted by % improvement (best first)
print('\n' + '='*120)
print('PROSPECTIVE METRIC PAIRS RANKED BY % SESSIONS WITH IMPROVEMENT')
print('='*120)

df_sorted = df_results.sort_values('pct_improvement', ascending=False)

display_cols = ['metric_pre', 'metric_post', 'n_sessions', 'pct_positive', 'pct_improvement', 'pct_significant', 'mean_effect', 'mean_improvement']
print(df_sorted[display_cols].to_string(index=False))


PROSPECTIVE METRIC PAIRS RANKED BY % SESSIONS WITH IMPROVEMENT
                                    metric_pre                                    metric_post  n_sessions  pct_positive  pct_improvement  pct_significant  mean_effect  mean_improvement
               phase_coherence_matrix_mean_mat                                             N1         305     66.557377        75.409836         7.868852     0.173518          0.183021
                                      salience                                             N1         304     67.763158        74.342105         9.539474     0.184763          0.202764
                                            N2                                             N1         303     68.646865        74.257426        11.551155     0.211078          0.213841
                                            N2                                       salience         303     64.026403        71.617162         8.580858     0.149037          0.193148
           

In [14]:
# Top 10 pairs
print('\n' + '='*120)
print('TOP 10 METRIC PAIRS')
print('='*120)

top_10 = df_sorted.head(10)
print(top_10[display_cols].to_string(index=False))

# Best pair summary
print('\n' + '='*120)
print('BEST PERFORMING METRIC PAIR')
print('='*120)
best = top_10.iloc[0]

print(f"\nMetric Pre:  {best['metric_pre']}")
print(f"Metric Post: {best['metric_post']}")
print(f"\nResults (n={int(best['n_sessions'])} sessions):")
print(f"  % Sessions with positive effect:     {best['pct_positive']:.1f}%")
print(f"  % Sessions with improvement:         {best['pct_improvement']:.1f}%")
print(f"  % Sessions with significance:        {best['pct_significant']:.1f}%")
print(f"  Mean effect size:                    {best['mean_effect']:.4f}")
print(f"  Mean improvement:                    {best['mean_improvement']:.4f}")


TOP 10 METRIC PAIRS
                                    metric_pre        metric_post  n_sessions  pct_positive  pct_improvement  pct_significant  mean_effect  mean_improvement
               phase_coherence_matrix_mean_mat                 N1         305     66.557377        75.409836         7.868852     0.173518          0.183021
                                      salience                 N1         304     67.763158        74.342105         9.539474     0.184763          0.202764
                                            N2                 N1         303     68.646865        74.257426        11.551155     0.211078          0.213841
                                            N2           salience         303     64.026403        71.617162         8.580858     0.149037          0.193148
                                            N2               N1_t         303     63.366337        70.957096        11.221122     0.123993          0.157962
                                     

In [15]:
# Save results
output_csv = metrics_root / 'prospective_metric_pairs_summary_hdEEG.csv'
df_results.to_csv(output_csv, index=False)

print(f'\nResults saved to: {output_csv}')


Results saved to: /Users/cbc/Documents/GitHub/fufo/notebook/DavideMomi/Revision/State_Dependent_Brain_Stimulation-main/data/df_results/hd-eeg_metrics/prospective_metric_pairs_summary_hdEEG.csv
